<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_3d_euler_maruyama_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Standalone Euler–Maruyama simulation of the 3D Cournot dynamics

This notebook isolates the direct stochastic baseline

$$
dX_t=b(X_t)\,dt+\sigma\,dW_t,
\qquad \sigma=0.1I_3,
$$

using

$$
X_{k+1}=X_k+h\,b(X_k)+\sqrt{h}\,\sigma\,\xi_k,
\qquad \xi_k\sim\mathcal N(0,I_3).
$$

It contains **no DTB network, tangent Jacobian, projection solve, coefficient vector, or score update**. The defaults match the independent Euler–Maruyama cloud in `cournot_3d_nonpotential_stochastic_mlp_dtb.ipynb`. The simulation stops cleanly before storing a non-finite proposal so its diagnostics and plots remain inspectable.


## 1. Imports and editable experiment controls


In [ ]:
from pathlib import Path
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

# These defaults reproduce the independent baseline in the current 3D notebook.
SEED = 2026
NOISE_SEED = SEED + 10_000
DIM = 3
N_PARTICLES = 2000
H = 0.005
T_FINAL = 1.0
N_STEPS = round(T_FINAL / H)
NOISE_STD = (0.1, 0.1, 0.1)
PRINT_EVERY = 10
SNAPSHOT_TIMES = (0.0, 0.2, 0.4, 0.6, 0.7)
SNAPSHOT_VIEW = (-1.0, 1.0)

# Optional local export in the Colab runtime.
SAVE_RESULTS = False
OUTPUT_DIR = Path('/content/cournot_3d_euler_maruyama_results')

DTYPE = torch.float32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COURNOT_B = 1.0
COURNOT_MU = 2.0

if not np.isclose(N_STEPS * H, T_FINAL):
    raise ValueError('T_FINAL must be an integer multiple of H.')
if len(NOISE_STD) != DIM:
    raise ValueError('NOISE_STD must contain one amplitude per coordinate.')

CONFIG = {
    'seed': SEED,
    'noise_seed': NOISE_SEED,
    'dimension': DIM,
    'particles': N_PARTICLES,
    'step_size': H,
    'steps': N_STEPS,
    'final_time': T_FINAL,
    'noise_std': list(NOISE_STD),
    'snapshot_view': list(SNAPSHOT_VIEW),
    'device': str(DEVICE),
    'dtype': str(DTYPE),
}
print(CONFIG)


## 2. Three-player Cournot drift

For player $i$, let $r_i=\sum_{j\ne i}x_j$. The exact unconstrained drift used by the current 3D experiment is

$$
b_i(x)=-2b x_i+2b\mu r_i-2b\mu r_i^2,
\qquad b=1,\quad \mu=2.
$$


In [ ]:
def game_velocity(x):
    """Unconstrained three-player Cournot payoff-gradient field."""
    if x.shape[-1] != DIM:
        raise ValueError(f'Expected states with last dimension {DIM}.')
    rivals = x.sum(dim=-1, keepdim=True) - x
    return (
        -2.0 * COURNOT_B * x
        + 2.0 * COURNOT_B * COURNOT_MU * rivals
        - 2.0 * COURNOT_B * COURNOT_MU * rivals.square()
    )


KNOWN_EQUILIBRIA = torch.tensor([
    [0.0, 0.0, 0.0],
    [3/8, 3/8, 3/8],
    [1/2, 1/2, 0.0],
    [1/2, 0.0, 1/2],
    [0.0, 1/2, 1/2],
], device=DEVICE, dtype=DTYPE)

equilibrium_residual = game_velocity(KNOWN_EQUILIBRIA).norm(dim=1).max()
print(f'Maximum equilibrium drift residual: {float(equilibrium_residual):.3e}')


## 3. Initialize the same uniform particle cloud and independent noise stream


In [ ]:
generator_device = DEVICE.type if DEVICE.type == 'cuda' else 'cpu'
initial_generator = torch.Generator(device=generator_device).manual_seed(SEED)
noise_generator = torch.Generator(device=generator_device).manual_seed(NOISE_SEED)

x_0 = torch.rand(
    (N_PARTICLES, DIM),
    device=DEVICE,
    dtype=DTYPE,
    generator=initial_generator,
)
x_k = x_0.clone()
sigma = torch.tensor(NOISE_STD, device=DEVICE, dtype=DTYPE)

print(
    f'Initial cloud: shape={tuple(x_k.shape)}, '
    f'min={float(x_k.min()):.6f}, max={float(x_k.max()):.6f}'
)


## 4. Run Euler–Maruyama only

Every step reports the finite-particle count, coordinate range, maximum absolute coordinate, negative-coordinate fraction, elapsed time, and ETA. The first non-finite proposal is diagnosed and retained separately; only finite states are stored in the history.


In [ ]:
def synchronize_device():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)


def finite_statistics(value):
    finite_components = torch.isfinite(value)
    finite_values = value[finite_components].to(torch.float64)
    if finite_values.numel() == 0:
        return float('nan'), float('nan'), float('nan'), float('nan')
    return (
        float(finite_values.min()),
        float(finite_values.max()),
        float(finite_values.abs().max()),
        float(torch.sqrt(torch.mean(finite_values.square()))),
    )


state_times = [0.0]
em_history = [x_k.detach().cpu().numpy()]
diagnostic_records = []
failure_step = None
failure_proposal = None

synchronize_device()
run_start = time.perf_counter()

for step in range(N_STEPS):
    step_start = time.perf_counter()
    drift_k = game_velocity(x_k)
    noise_k = torch.randn(
        x_k.shape,
        device=DEVICE,
        dtype=DTYPE,
        generator=noise_generator,
    )
    proposal = x_k + H * drift_k + math.sqrt(H) * sigma * noise_k
    synchronize_device()

    completed = step + 1
    finite_components = torch.isfinite(proposal)
    finite_particles = finite_components.all(dim=1)
    minimum, maximum, max_abs, rms = finite_statistics(proposal)
    step_seconds = time.perf_counter() - step_start
    elapsed_seconds = time.perf_counter() - run_start
    mean_step_seconds = elapsed_seconds / completed
    eta_seconds = mean_step_seconds * (N_STEPS - completed)
    negative_fraction = float((proposal[finite_components] < 0).float().mean())
    outside_fraction = float(
        ((proposal[finite_particles] < 0) | (proposal[finite_particles] > 1)).any(dim=1).float().mean()
    ) if bool(finite_particles.any()) else float('nan')

    record = {
        'step': completed,
        'time': completed * H,
        'finite_particles': int(finite_particles.sum()),
        'finite_components': int(finite_components.sum()),
        'minimum': minimum,
        'maximum': maximum,
        'max_abs': max_abs,
        'rms': rms,
        'negative_coordinate_fraction': negative_fraction,
        'outside_unit_box_particle_fraction': outside_fraction,
        'step_seconds': step_seconds,
        'elapsed_seconds': elapsed_seconds,
        'eta_seconds': eta_seconds,
    }
    diagnostic_records.append(record)

    should_report = completed == 1 or completed % PRINT_EVERY == 0
    if should_report or not bool(finite_particles.all()):
        print(
            f'EM {completed:4d}/{N_STEPS}  t={completed * H:.4f}  '
            f'finite={record["finite_particles"]}/{N_PARTICLES}  '
            f'range=[{minimum:.3e},{maximum:.3e}]  max|x|={max_abs:.3e}  '
            f'negative={negative_fraction:.3e}  step={step_seconds:.3f}s  '
            f'elapsed={elapsed_seconds:.2f}s  ETA={eta_seconds:.2f}s'
        )

    if not bool(finite_particles.all()):
        failure_step = completed
        failure_proposal = proposal.detach().cpu()
        failed_indices = torch.where(~finite_particles)[0].detach().cpu().tolist()
        print(f'\nStopped at the first non-finite proposal: step={failure_step}, t={failure_step * H:.4f}.')
        print(f'Non-finite particle indices (first 20): {failed_indices[:20]}')
        print(
            f'Non-finite components: {proposal.numel() - int(finite_components.sum())} '
            f'of {proposal.numel()}.'
        )
        break

    x_k = proposal.detach()
    state_times.append(completed * H)
    em_history.append(x_k.cpu().numpy())

em_history = np.stack(em_history)
state_times = np.asarray(state_times)
CONFIG['completed_finite_steps'] = len(state_times) - 1
CONFIG['failure_step'] = failure_step
CONFIG['wall_seconds'] = time.perf_counter() - run_start

if failure_step is None:
    print(f'Completed all {N_STEPS} Euler–Maruyama steps in {CONFIG["wall_seconds"]:.2f}s.')
else:
    print(
        f'Stored {CONFIG["completed_finite_steps"]} finite steps; '
        f'the proposal for step {failure_step} was non-finite.'
    )


## 5. Plot particle health and finite 3D snapshots

The particle snapshots use a fixed view window from `SNAPSHOT_VIEW`. Points outside that window are counted in each title rather than allowed to compress the visible cloud; the simulation history itself is never clipped.


In [ ]:
record_times = np.asarray([record['time'] for record in diagnostic_records])
max_abs_history = np.asarray([record['max_abs'] for record in diagnostic_records])
rms_history = np.asarray([record['rms'] for record in diagnostic_records])
negative_history = np.asarray([
    record['negative_coordinate_fraction'] for record in diagnostic_records
])
outside_history = np.asarray([
    record['outside_unit_box_particle_fraction'] for record in diagnostic_records
])
finite_particle_history = np.asarray([
    record['finite_particles'] for record in diagnostic_records
])

diagnostics_figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].semilogy(record_times, np.maximum(max_abs_history, 1e-30))
axes[0, 0].set(title='Maximum absolute coordinate', ylabel='max |x|')
axes[0, 1].semilogy(record_times, np.maximum(rms_history, 1e-30))
axes[0, 1].set(title='Particle-coordinate RMS', ylabel='RMS')
axes[1, 0].plot(record_times, negative_history, label='negative coordinates')
axes[1, 0].plot(record_times, outside_history, label='particles outside [0,1]³')
axes[1, 0].set(title='Departure from the initial strategy box', ylabel='fraction')
axes[1, 0].legend()
axes[1, 1].plot(record_times, finite_particle_history)
axes[1, 1].set(title='Finite particles', ylabel='count', ylim=(0, N_PARTICLES * 1.03))
for axis in axes.ravel():
    axis.set_xlabel('time')
    axis.grid(alpha=.25)
if failure_step is not None:
    for axis in axes.ravel():
        axis.axvline(failure_step * H, color='tab:red', linestyle='--', alpha=.7)
diagnostics_figure.suptitle('Standalone 3D Cournot Euler–Maruyama diagnostics')
plt.show()


available_snapshot_times = [
    value for value in SNAPSHOT_TIMES
    if value <= state_times[-1] + 1e-12 and np.isclose(round(value / H) * H, value)
]
columns = 3
rows = math.ceil(len(available_snapshot_times) / columns)
snapshots_figure = plt.figure(figsize=(5 * columns, 4.5 * rows), constrained_layout=True)
equilibria = KNOWN_EQUILIBRIA.detach().cpu().numpy()

for index, snapshot_time in enumerate(available_snapshot_times):
    state_index = round(snapshot_time / H)
    points = em_history[state_index]
    visible = np.all(
        (points >= SNAPSHOT_VIEW[0]) & (points <= SNAPSHOT_VIEW[1]), axis=1
    )
    axis = snapshots_figure.add_subplot(rows, columns, index + 1, projection='3d')
    axis.scatter(
        points[visible, 0], points[visible, 1], points[visible, 2], s=5, alpha=.35
    )
    axis.scatter(
        equilibria[:, 0], equilibria[:, 1], equilibria[:, 2],
        marker='x', s=60, color='tab:red', label='reported equilibria',
    )
    axis.set(
        title=(
            f't={snapshot_time:.2f}; shown {visible.sum()}/{len(points)} '
            f'in [{SNAPSHOT_VIEW[0]:g}, {SNAPSHOT_VIEW[1]:g}]³'
        ),
        xlabel='$x_1$', ylabel='$x_2$', zlabel='$x_3$',
        xlim=SNAPSHOT_VIEW, ylim=SNAPSHOT_VIEW, zlim=SNAPSHOT_VIEW,
    )
    axis.legend(loc='upper left')

for index in range(len(available_snapshot_times), rows * columns):
    axis = snapshots_figure.add_subplot(rows, columns, index + 1)
    axis.axis('off')
snapshots_figure.suptitle('Finite Euler–Maruyama particle clouds')
plt.show()


## 6. Optional local export


In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        OUTPUT_DIR / 'euler_maruyama_history.npz',
        times=state_times,
        particles=em_history,
        record_times=record_times,
        max_abs=max_abs_history,
        rms=rms_history,
        negative_coordinate_fraction=negative_history,
        outside_unit_box_particle_fraction=outside_history,
        finite_particle_count=finite_particle_history,
    )
    (OUTPUT_DIR / 'configuration.json').write_text(json.dumps(CONFIG, indent=2) + '\n')
    diagnostics_figure.savefig(
        OUTPUT_DIR / 'euler_maruyama_diagnostics.png', dpi=180, bbox_inches='tight'
    )
    snapshots_figure.savefig(
        OUTPUT_DIR / 'euler_maruyama_snapshots.png', dpi=180, bbox_inches='tight'
    )
    print('Saved standalone Euler–Maruyama results to:', OUTPUT_DIR)
else:
    print('SAVE_RESULTS=False: results remain in the Colab runtime only.')
